In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
from pathlib import Path

# Load .env from the root folder
BASE_DIR = Path.cwd().parent
load_dotenv(dotenv_path=BASE_DIR / ".env")

# Connect to Neon
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

def run_query(sql_query):
    """Execute a SQL query and return a Pandas DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(sql_query, conn)

print("✅ Environment ready. Connected to Neon.")

✅ Environment ready. Connected to Neon.


In [3]:
# RFM Analysis
query_rfm = """
WITH customer_metrics AS (
    SELECT 
        "CustomerID",
        MAX("InvoiceDate")::date AS last_purchase_date,
        COUNT(DISTINCT "Invoice") AS frequency,
        SUM("TotalAmount") AS monetary
    FROM retail_transactions
    GROUP BY "CustomerID"
),
rfm AS (
    SELECT 
        "CustomerID",
        (SELECT MAX("InvoiceDate")::date FROM retail_transactions) - last_purchase_date AS recency_days,
        frequency,
        monetary
    FROM customer_metrics
),
rfm_scores AS (
    SELECT 
        "CustomerID",
        recency_days,
        frequency,
        monetary,
        NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
        NTILE(5) OVER (ORDER BY frequency ASC) AS f_score,
        NTILE(5) OVER (ORDER BY monetary ASC) AS m_score
    FROM rfm
)
SELECT 
    "CustomerID",
    recency_days,
    frequency,
    ROUND(monetary::numeric, 2) AS monetary,
    r_score, f_score, m_score,
    CASE 
        WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'Champion'
        WHEN r_score >= 3 AND f_score >= 3 AND m_score >= 3 THEN 'Loyal'
        WHEN r_score >= 4 AND f_score <= 2 THEN 'New Customer'
        WHEN r_score <= 2 AND f_score >= 3 THEN 'At Risk'
        WHEN r_score <= 2 AND f_score <= 2 THEN 'Lost'
        ELSE 'Potential Loyalist'
    END AS segment
FROM rfm_scores;
"""

df_rfm = run_query(query_rfm)
print(f"Total customers segmented: {len(df_rfm)}")
print("\nSegment Distribution:")
print(df_rfm['segment'].value_counts())
print("\nRevenue by Segment:")
print(df_rfm.groupby('segment')['monetary'].agg(['count', 'sum', 'mean']).round(2).sort_values('sum', ascending=False))

Total customers segmented: 5878

Segment Distribution:
segment
Lost                  1519
Champion              1342
Loyal                 1095
At Risk                833
Potential Loyalist     641
New Customer           448
Name: count, dtype: int64

Revenue by Segment:
                    count          sum     mean
segment                                        
Champion             1342  11947714.02  8902.92
Loyal                1095   2485219.57  2269.61
At Risk               833   1615045.81  1938.83
Lost                 1519    631290.25   415.60
New Customer          448    388133.35   866.37
Potential Loyalist    641    307401.25   479.57
